# Keyword vs. Semantic vs. Hybrid Search — Decide by Experiment, Not Intuition

When building retrieval for a RAG system, the honest answer to "which search mode is best?" is: **it depends on your queries — so measure it.**

In a production quality-issue search system, users asked in two very different styles:

- **Keyword-style**: "TAC950 leak history" — exact terms matter
- **Symptom descriptions**: "water dripping under the unit after heating" — *same issue, different wording*

Keyword search misses the second kind; pure semantic search can lose precision on the first. This notebook reproduces that comparison on a synthetic dataset and shows why a **hybrid** configuration won, plus a final **business-priority re-ranking** layer — because users don't want "similar documents," they want "the document to act on now."

Pipeline: candidate methods → Recall@K / MRR → hybrid fusion → business re-ranking.

## 1. Setup & synthetic data

In [ ]:
%pip install openai pandas numpy scikit-learn -q

import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

client = OpenAI()  # requires OPENAI_API_KEY
EMB_MODEL = "text-embedding-3-small" 

In [ ]:
docs = pd.DataFrame([
    ("D01", "Heat pump refrigerant leak at valve joint", 0.15, "2025-06-01"),
    ("D02", "Water leakage under indoor unit due to clogged drain hose", 0.05, "2025-08-20"),
    ("D03", "Boiler ignition failure in cold weather", 0.30, "2024-11-02"),
    ("D04", "Condensate dripping beneath the unit after heating cycles", 0.08, "2025-09-14"),
    ("D05", "TAC950 controller firmware error code E13", 0.10, "2025-07-30"),
    ("D06", "Noise from circulation pump bearing wear", 0.45, "2023-12-11"),
    ("D07", "TAC950 sensor wiring corrosion causing intermittent shutdown", 0.20, "2025-05-19"),
    ("D08", "Gas water heater pilot light goes out repeatedly", 0.35, "2024-03-08"),
], columns=["id", "text", "rejection_rate", "reported_at"])

# Test queries with ground-truth relevant docs
queries = [
    ("TAC950 error history",                       {"D05", "D07"}),   # keyword-style
    ("water dripping under the unit after heating", {"D02", "D04"}),  # paraphrased symptom
    ("leak around refrigerant piping",              {"D01"}),
    ("unit shuts down on and off by itself",        {"D07"}),         # different wording
]

## 2. Three candidate retrieval methods

- **Keyword**: TF-IDF cosine (stand-in for BM25/simple search)
- **Semantic**: OpenAI embedding cosine similarity
- **Hybrid**: Reciprocal Rank Fusion (RRF) of both rankings — rank-based fusion avoids score-scale mismatch

In [ ]:
# Keyword scores
tfidf = TfidfVectorizer().fit(docs.text)
D_kw = tfidf.transform(docs.text)

def keyword_rank(q):
    sims = cosine_similarity(tfidf.transform([q]), D_kw)[0]
    return list(docs.id[np.argsort(-sims)])

# Semantic scores
def embed(texts):
    r = client.embeddings.create(model=EMB_MODEL, input=texts)
    return np.array([d.embedding for d in r.data])

D_sem = embed(list(docs.text))

def semantic_rank(q):
    sims = cosine_similarity(embed([q]), D_sem)[0]
    return list(docs.id[np.argsort(-sims)])

# Hybrid: Reciprocal Rank Fusion
def hybrid_rank(q, k=60):
    ranks = {}
    for ranking in (keyword_rank(q), semantic_rank(q)):
        for i, doc_id in enumerate(ranking):
            ranks[doc_id] = ranks.get(doc_id, 0) + 1.0 / (k + i + 1)
    return sorted(ranks, key=ranks.get, reverse=True)

## 3. Evaluate: Recall@3 and MRR

In [ ]:
def recall_at_k(ranking, relevant, k=3):
    return len(set(ranking[:k]) & relevant) / len(relevant)

def mrr(ranking, relevant):
    for i, d in enumerate(ranking, 1):
        if d in relevant:
            return 1.0 / i
    return 0.0

rows = []
for name, fn in [("keyword", keyword_rank), ("semantic", semantic_rank), ("hybrid", hybrid_rank)]:
    r3 = np.mean([recall_at_k(fn(q), rel) for q, rel in queries])
    m  = np.mean([mrr(fn(q), rel) for q, rel in queries])
    rows.append((name, round(r3, 3), round(m, 3)))

pd.DataFrame(rows, columns=["method", "recall@3", "MRR"])

Typical outcome: keyword wins on exact-term queries, semantic wins on paraphrases, and **hybrid is the only method strong on both** — which is exactly why it won the production comparison.

## 4. Business-priority re-ranking

Relevance is necessary but not sufficient. In production, reviewers wanted *actionable* cases first: recent, and historically likely to be accepted (low rejection rate). A final deterministic re-ranking layer combines the signals — kept **outside** the LLM on purpose, so ordering is explainable and stable.

In [ ]:
def business_rerank(ranking, top_n=5, w_rel=0.6, w_recent=0.25, w_accept=0.15):
    sub = docs.set_index("id").loc[ranking[:top_n]].copy()
    sub["rel"]     = np.linspace(1, 0, len(sub))                     # relevance rank -> score
    sub["recent"]  = (pd.to_datetime(sub.reported_at) - pd.Timestamp("2023-01-01")).dt.days
    sub["recent"] /= sub["recent"].max()
    sub["accept"]  = 1 - sub.rejection_rate
    sub["score"]   = w_rel*sub.rel + w_recent*sub.recent + w_accept*sub.accept
    return sub.sort_values("score", ascending=False)[["text", "rejection_rate", "reported_at", "score"]]

business_rerank(hybrid_rank("water dripping under the unit after heating"))

## Takeaways

- **Never pick a retrieval mode by intuition** — a 30-line evaluation harness settles it with your own query distribution.
- **Rank fusion (RRF) is a robust hybrid baseline** — no score normalization needed.
- **Keep business ranking deterministic** — separating LLM judgment from ranking logic makes results explainable to stakeholders and stable across runs.

*Based on a production quality-issue search system; data here is synthetic. Full write-up: [case study](../case-study-2-rag-search-system.md)*